In [5]:
import pandas as pd
import os

ref_path = '/content/Sony_Price_REF.csv'
data_path = '/content/Dataset Sony Camera(Lens not Included).csv'

if os.path.exists(ref_path) and os.path.exists(data_path):
    ref_df = pd.read_csv(ref_path)
    ref_df['score'] = 10
    df = pd.read_csv(data_path)

    display(ref_df.head())
    display(df.head())
    print(f'Reference rows: {len(ref_df)}')
    print(f'Dataset rows: {len(df)}')
else:
    print('Files still missing. Please ensure they are uploaded to /content/')

,model,release_year,age_year,shutter_claim,condition,price_used,score
0,Sony A5000,2014,12,100000,good,5000,10
1,Sony A5100,2014,12,100000,good,6500,10
2,Sony A6000,2014,12,100000,good,8000,10
3,Sony A6100,2019,7,100000,very_good,16500,10
4,Sony A6300,2016,10,100000,good,11500,10


,model,age_year,shutter_count,condition,has_box,price_used
0,Sony A6600,0,42098,very_good,0,24390
1,Sony A5100,1,8379,very_good,1,8000
2,Sony A7SIII,6,1026,excellent,0,61190
3,Sony A7RV,0,1684,very_good,0,97700
4,Sony A6600,6,4903,very_good,0,26100


Reference rows: 35
Dataset rows: 5000


In [10]:
import pandas as pd
import numpy as np
import os
import re

ref_path = '/content/Sony_Price_REF.csv'
data_path = '/content/Dataset Sony Camera(Lens not Included).csv'

if os.path.exists(ref_path) and os.path.exists(data_path):
    ref_df = pd.read_csv(ref_path)
    df = pd.read_csv(data_path)

    if 'score' not in ref_df.columns:
        ref_df['score'] = 10

    df['condition'] = df['condition'].astype(str).str.lower()
    df['shutter_count'] = pd.to_numeric(df['shutter_count'], errors='coerce').fillna(0)
    df['age_year'] = pd.to_numeric(df['age_year'], errors='coerce').fillna(0)

    ref_df['model_clean'] = ref_df['model'].str.replace('Sony ', '', regex=False)
    df['model_clean'] = df['model'].str.replace('Sony ', '', regex=False)

    merged_df = pd.merge(df, ref_df[['model_clean', 'price_used', 'score']], on='model_clean', how='left', suffixes=('', '_ref'))

    def calculate_scores(row):
        scores = {}
        cond_map = {'excellent': 10, 'very_good': 8, 'good': 6, 'fair': 4, 'poor': 2}
        scores['cond_score'] = cond_map.get(row['condition'], 5)
        scores['age_score'] = 10 if row['age_year'] < 3 else 5
        scores['shutter_score'] = 10 if row['shutter_count'] < 20000 else 5

        try:
            p_main = float(re.sub(r'[^0-9.]', '', str(row['price_used'])))
            p_ref = float(re.sub(r'[^0-9.]', '', str(row['price_used_ref'])))
            scores['price_score'] = 10 if p_main <= p_ref else 5
        except:
            scores['price_score'] = 5

        scores['box_score'] = 10 if row['has_box'] == 1 else 0
        return pd.Series(scores)

    score_cols = merged_df.apply(calculate_scores, axis=1)
    final_df = pd.concat([merged_df, score_cols], axis=1)
    print('Data processing complete. final_df is ready with 4-model ensemble compatibility.')
else:
    print('Error: CSV files missing in /content/')

Data processing complete. final_df is ready with 4-model ensemble compatibility.


In [6]:
from sklearn.model_selection import train_test_split

if 'final_df' in globals():
    available_features = [
        'age_year', 'shutter_count', 'has_box',
        'cond_score', 'age_score', 'shutter_score', 'price_score', 'box_score'
    ]
    features = [f for f in available_features if f in final_df.columns]
    X_ml = final_df[features].copy()
    y_price = final_df['price_used'].apply(lambda x: float(str(x).replace(',', '')) if pd.notnull(x) else 0)
    y_purchase = final_df['will_purchase'].astype(int)

    X_train, X_test, y_price_train, y_price_test = train_test_split(X_ml, y_price, test_size=0.2, random_state=42)
    _, _, y_purch_train, y_purch_test = train_test_split(X_ml, y_purchase, test_size=0.2, random_state=42)
    print(f'Using features: {features}')
    print(f'Data split successfully. Training on {len(X_train)} samples.')

In [17]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

def clean_val(val):
    try:
        res = float(str(val).replace(',', '').replace('-', '0'))
        return res if np.isfinite(res) else 0.0
    except:
        return 0.0

# Fix: ใช้ตัวแปร y_train และ X_train ที่ถูกสร้างไว้ใน cell ก่อนหน้า
y_price_train_clean = y_train.apply(clean_val).fillna(0.0)
y_price_test_clean = y_test.apply(clean_val).fillna(0.0)

X_train_numeric = X_train.apply(pd.to_numeric, errors='coerce').fillna(0.0)
X_test_numeric = X_test.apply(pd.to_numeric, errors='coerce').fillna(0.0)

price_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=42)
}

print('--- Price Prediction Model Results ---')
for name, model in price_models.items():
    model.fit(X_train_numeric, y_price_train_clean)
    preds = model.predict(X_test_numeric)
    r2 = r2_score(y_price_test_clean, preds)
    mae = mean_absolute_error(y_price_test_clean, preds)
    print(f'{name}: R2 Score = {r2:.4f}, MAE = {mae:.2f}')

--- Price Prediction Model Results ---
Linear Regression: R2 Score = 0.1594, MAE = 27523.43
Random Forest Regressor: R2 Score = 0.1629, MAE = 25817.42
Gradient Boosting Regressor: R2 Score = 0.2600, MAE = 25114.24


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
purchase_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest Classifier': RandomForestClassifier(random_state=42),
    'Gradient Boosting Classifier': GradientBoostingClassifier(random_state=42)
}

print('--- Purchase Prediction Accuracy Results ---')
for name, model in purchase_models.items():
    model.fit(X_train_numeric, y_purch_train)
    preds = model.predict(X_test_numeric)
    acc = accuracy_score(y_purch_test, preds)
    print(f'{name} Accuracy: {acc*100:.2f}%')

In [ ]:
from sklearn.ensemble import VotingRegressor, VotingClassifier

reg_ensemble = VotingRegressor(estimators=[
    ('lr', LinearRegression()),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingRegressor(random_state=42))
])
reg_ensemble.fit(X_train_numeric, y_price_train_clean)
reg_preds = reg_ensemble.predict(X_test_numeric)
clf_ensemble = VotingClassifier(estimators=[
    ('log_reg', LogisticRegression(max_iter=1000)),
    ('rf_clf', RandomForestClassifier(random_state=42)),
    ('gb_clf', GradientBoostingClassifier(random_state=42))
], voting='hard')
clf_ensemble.fit(X_train_numeric, y_purch_train)
clf_preds = clf_ensemble.predict(X_test_numeric)

print('--- Ensemble Model (3 Types Combined) ---')
print(f'Regression (Price) R2 Score: {r2_score(y_price_test_clean, reg_preds):.4f}')
print(f'Classification (Purchase) Accuracy: {accuracy_score(y_purch_test, clf_preds)*100:.2f}%')

current_price_model = reg_ensemble
current_purchase_model = clf_ensemble

In [12]:
import joblib
import pandas as pd

try:
    model = joblib.load('Sony_Camera_Predict.joblib')
    test_data = pd.DataFrame([
        [1, 5000, 1, 10, 10, 10, 10, 10],
        [8, 85000, 0, 6, 2, 4, 6, 0]
    ], columns=['age_year', 'shutter_count', 'has_box', 'cond_score', 'age_score', 'shutter_score', 'price_score', 'box_score'])

    predictions = model.predict(test_data)

    print('--- ผลการทดสอบ ---')
    print(f'ตัวอย่างที่ 1: {predictions[0]:,.2f} บาท')
    print(f'ตัวอย่างที่ 2: {predictions[1]:,.2f} บาท')

except Exception as e:
    print(f'ไม่พบไฟล์โมเดลหรือเกิดข้อผิดพลาด: {e}')

--- ผลการทดสอบจาก Ensemble Model (4 Types) ---
ตัวอย่างที่ 1: 75,207.43 บาท
ตัวอย่างที่ 2: 11,333.45 บาท

โมเดลที่ใช้: VotingRegressor (RF + GB + LinearReg + ExtraTrees)


In [16]:
import joblib
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.linear_model import LinearRegression

if 'final_df' in globals():
    features = ['age_year', 'shutter_count', 'has_box', 'cond_score', 'age_score', 'shutter_score', 'price_score', 'box_score']
    X_ml = final_df[features].apply(pd.to_numeric, errors='coerce').fillna(0.0)

    def clean_currency(value):
        clean_str = re.sub(r'[^0-9.]', '', str(value))
        try: return float(clean_str)
        except: return 0.0

    y_price = final_df['price_used'].apply(clean_currency)
    X_train, X_test, y_train, y_test = train_test_split(X_ml, y_price, test_size=0.2, random_state=42)


    final_model = VotingRegressor([
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
        ('gb', GradientBoostingRegressor(random_state=42)),
        ('lr', LinearRegression()),
        ('et', ExtraTreesRegressor(n_estimators=100, random_state=42))
    ])

    final_model.fit(X_train, y_train)
    joblib.dump(final_model, 'Sony_Camera_Predict.joblib')
    print('Model trained with 4 types: RF, GB, LR, ET saved.')
else:
    print('Error: final_df missing. Please run cell 82f8641e first.')

Model trained with 4 types: RF, GB, LR, ET saved.


In [14]:
from google.colab import files
import os

if os.path.exists('Sony_Camera_Predict.joblib'):
    print('Initiating download...')
    files.download('Sony_Camera_Predict.joblib')
else:
    print('Error: Model file Sony_Camera_Predict.joblib not found. Please ensure the training cell (c21afac8) ran successfully.')

Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>